# Notebook Laboratorio 3 Bases de Datos Avanzadas - Neo4J

Descarga de librería

In [1]:
%pip install neo4j

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Librerías y dependencias

In [2]:
import pandas as pd
from neo4j import GraphDatabase

## 2. Conexión y creación de BD Neo4J

In [15]:
URI = "bolt://localhost:7687"
AUTH = ("neo4j", "password123")
driver = GraphDatabase.driver(URI, auth=AUTH)
session = driver.session()

Si quieren trabajar con ver el grafo deben abrir Neo4j Browser. Este esta en http://localhost:7474/browser/.

Para entrar deben ingresar el usuario y contraseña, lo que esta en el AUTH

## 3. Procesamiento y Poblamiento de la BD Neo4J

In [16]:
# Celda de limpieza: Borra todos los nodos y relaciones del grafo
query_limpieza = "MATCH (n) DETACH DELETE n"

session.run(query_limpieza)
print("¡Grafo limpiado por completo! Listo para probar el nuevo ciclo for.")

¡Grafo limpiado por completo! Listo para probar el nuevo ciclo for.


In [ ]:
df = pd.read_csv('normativas/normativas_clasificadas_IA.csv')
df.fillna("", inplace=True)

reglas_referencia = {
    "Resolución Exenta N° 176 de 2020": "Resolución 176 de 2020",
    "Resolución Exenta N° 76 de 2021": "Resolución 76 de 2021",
    "Resolución Exenta N° 79 de 2025": "Resolución 79 de 2025",
    "Resolución N° 59 de 2025": "Resolución 59 de 2025",
    "Circular N° 38 de 2025": "Circular 38 de 2025",
    "Articulo 68 del Código Tributario": "Articulo 68 del Código Tributario"
}

reglas_palabras = {
    "boleta": "Contiene 'boleta'",
    "comprobante electrónico": "Contiene 'comprobante electrónico'",
    "registro de compra": "Contiene 'registro de compra'",
    "registro de venta": "Contiene 'registro de venta'",
    "cumplimiento tributario": "Contiene 'cumplimiento tributario'",
    "inicio de actividades": "Contiene 'inicio de actividades'",
    "medios de pago electrónicos": "Contiene 'medios de pago electrónicos'",
    "pos": "Contiene 'POS'",
    "p.o.s": "Contiene 'P.O.S'",
    "puntos de venta": "Contiene 'puntos de venta'",
    "operadores y administradores": "Contiene 'operadores y administradores'",
    "comercio electrónico": "Contiene 'comercio electrónico'"
}

for index, row in df.iterrows():
    nombre = str(row['name'])
    desc = str(row['description'])
    fuente = str(row['fuente'])
    url = str(row['url'])
    tipo_doc = str(row['tipo_documento'])
    cuerpo = str(row['cuerpo'])
    relevancia = str(row['relevancia'].strip())
    if relevancia == "No Relevante":
        relevancia = "NoRelevante"
    elif relevancia == "Relevante":
        relevancia = "Relevante"
    explicacion = str(row['explicacion'])
    
    texto_completo = f"{nombre} {desc} {cuerpo} {explicacion}"
    texto_lower = texto_completo.lower()
    reglas_activadas = []
    
    for ref, regla in reglas_referencia.items():
        if ref.lower() in texto_lower:
            reglas_activadas.append((regla, f"Se detectó la referencia: {ref}"))
            
    for palabra, regla in reglas_palabras.items():
        if palabra.lower() in texto_lower:
            reglas_activadas.append((regla, f"Se detectó la palabra clave: {palabra}"))
            
    labels = ["Normativa"]
    if "Circular" in tipo_doc: 
        labels.append("Circular")
    if "Resolución" in tipo_doc or "Resolucion" in tipo_doc: 
        labels.append("Resolucion")
    
    labels.append(relevancia)

    if relevancia == "Relevante":
        if len(reglas_activadas) > 0:
            labels.append("ExplicacionValida")
        else:
            labels.append("ExplicacionDebil")
            labels.append("RequiereRevision")

    else:  
        if len(reglas_activadas) == 0:
            labels.append("ExplicacionValida")
        else:
            labels.append("RequiereRevision")
            
    labels_str = ":".join(labels)
    
    query_base = f"""
    MERGE (agente:AgenteIANormativo {{nombre: 'Agente IA Normativo'}})
    MERGE (f:Fuente {{nombre: $fuente}})
    MERGE (n:{labels_str} {{nombre: $nombre}})
      SET n.descripcion = $desc, 
          n.url = $url, 
          n.cuerpo = $cuerpo
          
    MERGE (n)-[:EMITIDA_POR]->(f)
    MERGE (n)-[:CLASIFICADA_POR]->(agente)
    
    CREATE (exp:ExplicacionIA {{texto: $explicacion}})
    MERGE (n)-[:TIENE_EXPLICACION]->(exp)
    """
    session.run(query_base, fuente=fuente, nombre=nombre, desc=desc, url=url, cuerpo=cuerpo, explicacion=explicacion)
    
    for regla_nombre, evidencia in reglas_activadas:
        query_reglas = """
        MATCH (n:Normativa {nombre: $nombre})
        MERGE (r:ReglaDeNegocio {nombre: $regla_nombre})
        MERGE (n)-[:ACTIVA_REGLA]->(r)
        
        CREATE (ev:EvidenciaTextual {texto: $evidencia})
        MERGE (n)-[:RESPALDADA_POR]->(ev)
        """
        session.run(query_reglas, nombre=nombre, regla_nombre=regla_nombre, evidencia=evidencia)

## 4. Consultas obligatorias

4.1 Consulta de clasificación general: visualizar normativas Relevantes y No Relevantes con nombre, tipo documental, fuente, descripción y explicación IA.

In [11]:
# Consulta 1: Clasificación general
q1 = """
MATCH (n:Normativa)-[:TIENE_EXPLICACION]->(e:ExplicacionIA)
MATCH (n)-[:EMITIDA_POR]->(f:Fuente)
RETURN n.nombre AS Normativa, 
       labels(n) AS Etiquetas, 
       f.nombre AS Fuente, 
       n.descripcion AS Descripcion,
       e.texto AS Explicacion_IA
LIMIT 10
"""
res1 = session.run(q1)
df1 = pd.DataFrame([r.data() for r in res1])
df1

,Normativa,Etiquetas,Fuente,Descripcion,Explicacion_IA
0,Resolución Exenta SII N° 77 del 26 de Junio de...,"[Normativa, Relevante, ExplicacionValida, Reso...",Fuente: Subdireccion Juridica,Establece procedimiento para la obtención de R...,La normativa se relaciona directamente con el ...
1,Resolución Exenta SII N° 67 del 12 de Junio de...,"[Normativa, NoRelevante, Resolucion]",Fuente: Subdireccion Juridica,Establece medidas para asegurar el cumplimient...,La normativa se centra en el deber de reserva ...
2,Resolución Exenta SII N° 64 del 27 de Mayo del...,"[Normativa, NoRelevante, Resolucion]",Fuente: Subdireccion Juridica,Delega facultad que indica en la funcionaria q...,La normativa se refiere a la delegación de fac...
3,Resolución Exenta SII N° 59 del 06 de Mayo del...,"[Normativa, NoRelevante, Resolucion]",Fuente: Subdireccion Juridica,Establece procedimiento para efectuar denuncia...,La normativa se centra en establecer un proced...
4,Resolución Exenta SII N° 58 del 06 de Mayo del...,"[Normativa, NoRelevante, Resolucion]",Fuente: Subdireccion Juridica,Establece parámetros objetivos para determinar...,No cumple reglas de negocio. La normativa se c...
5,Resolución Exenta SII N° 49 del 17 de Abril de...,"[Normativa, NoRelevante, Resolucion]",Fuente: Subdireccion Juridica,Tramitación de las autodenuncias conforme al a...,La normativa se centra en el procedimiento de ...
6,Resolución Exenta SII N° 29 del 06 de Marzo de...,"[Normativa, NoRelevante, Resolucion]",Fuente: Subdireccion Juridica,"Establece criterios para la proposición, negoc...",La normativa se centra en establecer criterios...
7,Resolución Exenta SII N° 16 del 30 de Enero de...,"[Normativa, NoRelevante, Resolucion]",Fuente: Subdireccion Juridica,Delega facultad que indica en la funcionaria q...,La normativa se refiere a la delegación de fac...
8,Resolución Exenta SII N° 10 del 16 de Enero de...,"[Normativa, NoRelevante, Resolucion]",Fuente: Subdireccion Juridica,Delega facultad que indica en la funcionaria q...,La normativa se refiere a la delegación de fac...
9,Circular N° 7 del 16 de Enero del 2025,"[Normativa, Circular, NoRelevante]",Fuente: Subdireccion Juridica,Imparte instrucciones sobre los artículos 62 y...,La normativa se centra en el acceso a informac...


Para ver en Neo4j Browser

In [ ]:
MATCH (n:Normativa)-[r1:TIENE_EXPLICACION]->(e:ExplicacionIA)
MATCH (n)-[r2:EMITIDA_POR]->(f:Fuente)
RETURN n, r1, e, r2, f
LIMIT 10

4.2 Consulta explicativa de una normativa específica: mostrar clasificación IA, explicación, 
reglas activadas y evidencia textual asociada. 

Buscamos los nombres de las normativas relevantes

In [18]:
# Consulta rápida para ver los nombres reales de tus normativas relevantes
q_nombres = """
MATCH (n:Relevante)
RETURN n.nombre AS Nombre_Real
LIMIT 5
"""
df_nombres = pd.DataFrame([r.data() for r in session.run(q_nombres)])
df_nombres

,Nombre_Real
0,Circular N° 12 del 30 de Enero del 2025
1,Circular N° 19 del 06 de Marzo del 2025
2,Circular N° 2 del 02 de Enero del 2025
3,Circular N° 32 del 17 de Abril del 2025
4,Circular N° 33 del 17 de Abril del 2025


In [20]:
q2 = """
MATCH (n:Normativa {nombre: $nombre_buscar})
MATCH (n)-[:TIENE_EXPLICACION]->(e:ExplicacionIA)
OPTIONAL MATCH (n)-[:ACTIVA_REGLA]->(r:ReglaDeNegocio)
OPTIONAL MATCH (n)-[:RESPALDADA_POR]->(ev:EvidenciaTextual)
RETURN n.nombre AS Normativa, 
       labels(n) AS Clasificacion,
       e.texto AS Explicacion, 
       collect(DISTINCT r.nombre) AS Reglas_Activadas, 
       collect(DISTINCT ev.texto) AS Evidencias_Encontradas
"""
df2 = pd.DataFrame([r.data() for r in session.run(q2, nombre_buscar="Circular N° 12 del 30 de Enero del 2025")])
df2

,Normativa,Clasificacion,Explicacion,Reglas_Activadas,Evidencias_Encontradas
0,Circular N° 12 del 30 de Enero del 2025,"[Normativa, Circular, Relevante, ExplicacionVa...",La normativa aborda modificaciones en la Ley s...,"[Contiene 'comercio electrónico', Contiene 'PO...",[Se detectó la palabra clave: comercio electró...


Neo4j Browser (Cambiar el nombre por un nombre que sale en la tabla de arriba)

In [ ]:
MATCH (n:Normativa {nombre: "<nombre>"})-[r1:TIENE_EXPLICACION]->(e:ExplicacionIA)
OPTIONAL MATCH (n)-[r2:ACTIVA_REGLA]->(r:ReglaDeNegocio)
OPTIONAL MATCH (n)-[r3:RESPALDADA_POR]->(ev:EvidenciaTextual)
RETURN n, r1, e, r2, r, r3, ev

4.3 Consulta de normativas relevantes con respaldo de negocio: identificar normativas 
Relevantes que activan reglas de negocio y cuentan con evidencia textual. 

In [13]:
q3 = """
MATCH (n:Relevante)-[:ACTIVA_REGLA]->(r:ReglaDeNegocio)
MATCH (n)-[:RESPALDADA_POR]->(ev:EvidenciaTextual)
RETURN n.nombre AS Normativa, 
       count(DISTINCT r) AS Cantidad_Reglas, 
       collect(DISTINCT r.nombre) AS Reglas
"""
df3 = pd.DataFrame([r.data() for r in session.run(q3)])
df3

,Normativa,Cantidad_Reglas,Reglas
0,Circular N° 12 del 30 de Enero del 2025,4,"[Contiene 'comercio electrónico', Contiene 'PO..."
1,Circular N° 19 del 06 de Marzo del 2025,3,"[Contiene 'POS', Contiene 'medios de pago elec..."
2,Circular N° 2 del 02 de Enero del 2025,4,"[Contiene 'operadores y administradores', Cont..."
3,Circular N° 32 del 17 de Abril del 2025,3,"[Contiene 'POS', Contiene 'inicio de actividad..."
4,Circular N° 33 del 17 de Abril del 2025,5,"[Contiene 'comercio electrónico', Contiene 'PO..."
5,Circular N° 38 del 30 de Abril del 2025,4,"[Contiene 'POS', Contiene 'medios de pago elec..."
6,Circular N° 39 del 30 de Abril del 2025,5,"[Contiene 'comercio electrónico', Contiene 'PO..."
7,Resolución Exenta SII N° 11 del 17 de Enero de...,3,"[Contiene 'POS', Contiene 'inicio de actividad..."
8,Resolución Exenta SII N° 12 del 17 de Enero de...,4,"[Contiene 'POS', Contiene 'medios de pago elec..."
9,Resolución Exenta SII N° 14 del 30 de Enero de...,2,"[Contiene 'POS', Contiene 'cumplimiento tribut..."


Neo4j Browser

In [ ]:
MATCH (n:Relevante)-[r1:ACTIVA_REGLA]->(r:ReglaDeNegocio)
MATCH (n)-[r2:RESPALDADA_POR]->(ev:EvidenciaTextual)
RETURN n, r1, r, r2, ev

4.4 Consulta de posibles inconsistencias: detectar No Relevantes que activan reglas de 
negocio, o Relevantes que no activan reglas. 

In [ ]:
q4 = """
MATCH (n:Normativa)
WHERE (n:NoRelevante AND (n)-[:ACTIVA_REGLA]->()) 
   OR (n:Relevante AND NOT (n)-[:ACTIVA_REGLA]->())
RETURN n.nombre AS Normativa_Sospechosa, 
       labels(n) AS Clasificacion_IA, 
       n.descripcion AS Descripcion
"""
df4 = pd.DataFrame([r.data() for r in session.run(q4)])
df4

,Normativa_Sospechosa,Clasificacion_IA,Descripcion
0,Circular N° 1 del 02 de Enero del 2025,"[Normativa, Circular, NoRelevante]",Imparte instrucciones sobre normas del Código ...
1,Circular N° 10 del 30 de Enero del 2025,"[Normativa, Circular, NoRelevante]",Imparte instrucciones sobre las modificaciones...
2,Circular N° 11 del 30 de Enero del 2025,"[Normativa, Circular, NoRelevante]",Imparte instrucciones sobre las modificaciones...
3,Circular N° 13 del 07 de Febrero del 2025,"[Normativa, Circular, NoRelevante]",Imparte instrucciones sobre el artículo 100 se...
4,Circular N° 14 del 10 de Febrero del 2025,"[Normativa, Circular, NoRelevante]",Tablas de impuesto único de segunda categoría ...
...,...,...,...
85,Resolución Exenta SII N° 78 del 26 de Junio de...,"[Normativa, NoRelevante, Resolucion]",Autoriza acceso a los Servicios de Interoperab...
86,Resolución Exenta SII N° 80 del 26 de Junio de...,"[Normativa, NoRelevante, Resolucion]",Establece procedimiento para presentar la soli...
87,Resolución Exenta SII N° 81 del 30 de Junio de...,"[Normativa, NoRelevante, Resolucion]",Modifica fecha de entrada en vigencia de la Re...
88,Resolución Exenta SII N° 82 del 03 de Julio de...,"[Normativa, NoRelevante, Resolucion]",Aprueba Convenio de Intercambio de Información...


Neo4j Browser

In [ ]:
MATCH (n:Normativa)
WHERE (n:NoRelevante AND (n)-[:ACTIVA_REGLA]->()) 
   OR (n:Relevante AND NOT (n)-[:ACTIVA_REGLA]->())
OPTIONAL MATCH (n)-[r1:ACTIVA_REGLA]->(r:ReglaDeNegocio)
OPTIONAL MATCH (n)-[r2:TIENE_EXPLICACION]->(e:ExplicacionIA)
RETURN n, r1, r, r2, e

4.5 Consulta de revisión humana: identificar normativas con explicación débil, insuficiente o 
poco alineada con las reglas de negocio. 

In [16]:
q5 = """
MATCH (n:RequiereRevision)-[:TIENE_EXPLICACION]->(e:ExplicacionIA)
RETURN n.nombre AS Normativa, 
       labels(n) AS Etiquetas_Actuales, 
       e.texto AS Explicacion_IA
"""
df5 = pd.DataFrame([r.data() for r in session.run(q5)])
df5

Received notification from DBMS server: <GqlStatusObject gql_status='01N50', status_description='warn: label does not exist. The label `RequiereRevision` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=2, column=10, offset=10>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 2, 'column': 10}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (n:RequiereRevision)-[:TIENE_EXPLICACION]->(e:ExplicacionIA)\nRETURN n.nombre AS Normativa, \n       labels(n) AS Etiquetas_Actuales, \n       e.texto AS Explicacion_IA\n'


""


Neo4j Browser

In [ ]:
MATCH (n:RequiereRevision)-[r:TIENE_EXPLICACION]->(e:ExplicacionIA)
RETURN n, r, e

In [17]:
# Consulta de control: Cuenta cuántos nodos hay por cada etiqueta existente
q_control = """
MATCH (n:Normativa)
RETURN labels(n) AS Etiquetas, count(n) AS Total
"""
df_control = pd.DataFrame([r.data() for r in session.run(q_control)])
df_control

,Etiquetas,Total
0,"[Normativa, Circular, NoRelevante]",39
1,"[Normativa, Circular, Relevante, ExplicacionVa...",7
2,"[Normativa, NoRelevante, Resolucion]",70
3,"[Normativa, Relevante, ExplicacionValida, Reso...",13


## 5. Auditoria Simulada

### 5.1 Tabla de Auditoría de Iniciativas (Evaluación del Agente IA)

De acuerdo a los requerimientos del laboratorio, se seleccionan 5 normativas clave integradas en el grafo de Neo4j (tanto clasificadas como *Relevantes* como *No Relevantes*) para contrastar la decisión del modelo automatizado con el criterio experto del grupo.

La siguiente celda realiza una consulta Cypher dinámica para extraer el estado del grafo y le añade las columnas de **Juicio del Grupo** y **Justificación** requeridas para la auditoría humana simulada:

In [ ]:
# 1. Definir explícitamente las 5 normativas a auditar presentes en el dataset
normativas_a_auditar = [
    "Circular N° 12 del 30 de Enero del 2025",
    "Circular N° 19 del 06 de Marzo del 2025",
    "Resolución Exenta SII N° 77 del 26 de Junio de 2025",
    "Resolución Exenta SII N° 67 del 12 de Junio de 2025",
    "Resolución Exenta SII N° 58 del 06 de Mayo del 2025"
]

# 2. Consulta Cypher para extraer la información estructurada desde Neo4j
query_auditoria = """
MATCH (n:Normativa)
WHERE n.nombre IN $nombres
MATCH (n)-[:TIENE_EXPLICACION]->(e:ExplicacionIA)
OPTIONAL MATCH (n)-[:ACTIVA_REGLA]->(r:ReglaDeNegocio)
OPTIONAL MATCH (n)-[:RESPALDADA_POR]->(ev:EvidenciaTextual)
RETURN n.nombre AS Normativa_Revisada,
       [lbl IN labels(n) WHERE lbl <> 'Normativa'] AS Clasificacion_IA,
       collect(DISTINCT r.nombre) AS Reglas_Activadas,
       collect(DISTINCT ev.texto) AS Evidencia_Textual
"""

res_auditoria = session.run(query_auditoria, nombres=normativas_a_auditar)
df_auditoria = pd.DataFrame([r.data() for r in res_auditoria])

# 3. Mapeo del Juicio Humano del Grupo y Justificación para cada iniciativa
evaluacion_humana = {
    "Circular N° 12 del 30 de Enero del 2025": {
        "Juicio": "Validada",
        "Justificacion": "La clasificación de la IA es correcta. El documento aborda explícitamente modificaciones operacionales críticas de comercio electrónico y terminales POS."
    },
    "Circular N° 19 del 06 de Marzo del 2025": {
        "Juicio": "Validada",
        "Justificacion": "Clasificación consistente con el negocio. Activa de manera correcta las reglas de medios de pago electrónicos basándose en el cuerpo normativo."
    },
    "Resolución Exenta SII N° 77 del 26 de Junio de 2025": {
        "Juicio": "Validada",
        "Justificacion": "El agente IA identificó con precisión la relación con los procesos de inicio de actividades y cumplimiento tributario vigentes."
    },
    "Resolución Exenta SII N° 67 del 12 de Junio de 2025": {
        "Juicio": "Validada",
        "Justificacion": "Correctamente catalogada como No Relevante. Su enfoque está limitado estrictamente al deber de reserva interna institucional del servicio."
    },
    "Resolución Exenta SII N° 58 del 06 de Mayo del 2025": {
        "Juicio": "Requiere más antecedentes",
        "Justificacion": "Aunque la IA la clasificó como No Relevante por falta de palabras clave explícitas, la descripción técnica de los parámetros objetivos amerita un análisis legal manual extendido."
    }
}

# 4. Incorporar las columnas del criterio humano al DataFrame de resultados
df_auditoria['Juicio del Grupo'] = df_auditoria['Normativa_Revisada'].map(lambda x: evaluacion_humana.get(x, {}).get('Juicio', 'No Evaluado'))
df_auditoria['Justificación'] = df_auditoria['Normativa_Revisada'].map(lambda x: evaluacion_humana.get(x, {}).get('Justificacion', ''))

# Reordenar columnas para cumplir exactamente con la estructura solicitada
columnas_ordenadas = ['Normativa_Revisada', 'Clasificacion_IA', 'Reglas_Activadas', 'Evidencia_Textual', 'Juicio del Grupo', 'Justificación']
df_auditoria = df_auditoria[columnas_ordenadas]

# 5. Desplegar la matriz de auditoría
pd.set_option('display.max_colwidth', None)
df_auditoria

**Nota de Integración:** A través de esta simulación se logra validar el comportamiento del agente de IA en un 80% de aciertos directos (*Validadas*), aislando un caso crítico (*Requiere más antecedentes*) donde las reglas por palabras clave resultaron insuficientes frente a la semántica compleja de la resolución.

## 6. Pruebas de Consistencia

## 7. Pruebas de Disponibilidad